# Chapter 1

### Parameters in a neural network

<center><img src="images/01.01.png"  style="width: 400px, height: 300px;"/></center>

# Chapter 2

### Multi label vs multi class

<center><img src="images/02.01.png"  style="width: 400px, height: 300px;"/></center>


### Can we solve multi-class problem without using softmax?

- use 'sigmoid' activation function in the output layer. This will act as one-vs-rest classification (2-class classification) problem
- Since this becomes a two-class classification problem now, use 'binary_crossentropy' as the loss function
- You can use 'adam' as optimizer
- This approach is preferable for multi-label classification problem instead of multi-class classification problem

<center><img src="images/02.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/02.03.png"  style="width: 400px, height: 300px;"/></center>


### Keras Neural Network

```
data = pd.read_csv('dataset.csv')
sns.pairplot(data, hue='target')  # Good to explore the dataset
plt.show()
X = data.drop(['target'], axis=1).values 
n_cols = X.shape[1] # Get the number of features other than target column
# For categorical target variable, you need something like one-hot-encoding
from tensorflow.keras.utils import to_categorical
y = to_categorical(data['target']) # Use this for one-hot-encoding if target is a class and it is a classification problem
from tensorflow.keras.layers import Dense
from tensorflow.keras.models import Sequential
def create_model(optimizer='adam', activation='relu', nl=1,nn=256):
  model = Sequential()
  # for i in range(nl): # you can also automate this process of creating layers and neurons with specified arguments
  # # Layers have nn neurons
  #   model.add(Dense(nn, activation=activation))
  # Start with the first hidden layer, where information comes in shape (dataset column, any number of rows)
  model.add(Dense(100, activation=activation, input_shape = (n_cols,))) # First hidden layer, where values directly come from dataset of n features, with unknown number of datapoints
  model.add(Dense(100, activation=activation, kernel_initializer='normal')) # Second hidden layer
  model.add(Dense(100, activation=activation)) # Third hidden layer
  from tensorflow.keras.layers import BatchNormalization
  model.add(BatchNormalization()) # Add batch normalization for the outputs of the layer above
  # use 'softmax' for multi-class classification, 'sigmoid' for binary or multi-label classification
  model.add(Dense(3, activation='softmax')) # Output layer, 3 nodes for 3 class prediction
  my_optimizer = SGD(lr=lr) # # You can also use custom optimizer like this
  # For loss function, use 'categorical_crossentropy' for multiclass, 'binary_crossentropy' for binary or multi-label, 'mean_squared_error'  for regression
  model.compile(optimizer=optimizer, loss='categorical_crossentropy', metrics=['accuracy']) # optimizer = my_optimizer
  return model
# K-fold is costly for neural network, so use validation data as a wise choice to get best validation score with early stopping
from tensorflow.keras.callbacks import EarlyStopping
early_stopping_monitor = EarlyStopping(monitor='val_loss', patience=2) # Terminates once validation loss stops improving
# Instantiate a model checkpoint callback, this will automatically save the model when best result is produced
from keras.callbacks import ModelCheckpoint
model_save = ModelCheckpoint('best_model.hdf5', save_best_only=True)
# When fitting, you can use validation data using : "validation_data=(X_test, y_test)" or directly use validation_split
model = create_model()
model_1_training = model.fit(X_train, y_train, validation_split=0.3, epochs=20, batch_size=128, callbacks = [early_stopping_monitor, model_save], verbose=0)
predictions = model.predict(X_test)
probability_true = predictions[:,1] 

from tensorflow.keras.models import load_model
init_weights = model.get_weights() # Save model weights
model.save('model_file.h5') # Saving  model
first_layer = model.layers[0] # Acessing the first layer of a Keras model
print(first_layer.input) # Printing the layer input (this is a tensorflow tensor object)
print(first_layer.output) # Printing the layer output  (this is a tensorflow tensor object)
print(first_layer.weights) # Printing the layer weights (this is a tensorflow variable object, which is updatable)
my_model = load_model('model_file.h5') # loading model
my_model.summary() # See model summary
history = model_1_training # Extracting fitting history
print(history.params) # See all parameters
print(history.history.keys()) # See what you can extract, eg :  val_loss = history.history['val_loss'], val_accuracy = history.history['val_accuracy']
accuracy = model.evaluate(X_test, y_test)[1] # Evaluate
preds = model.predict(test_set) # Predict 
# Extract the position of highest probability from each pred vector (For classification of multi-class problem)
preds_chosen = [np.argmax(pred) for pred in preds]

### VISUALIZE KERAS LEARNING CURVE with plot_loss(loss,val_loss) and plot_accuracy(acc,val_acc) using fitting/training history

h_callback = model.fit(X_train, y_train, epochs = 25, validation_data=(X_test, y_test)) # Train your model and save its history
plot_loss(h_callback.history['loss'], h_callback.history['val_loss']) # Plot train vs test loss during training
plot_accuracy(h_callback.history['accuracy'], h_callback.history['val_accuracy']) # Plot train vs test accuracy during training

# Visualize if train size increase accuracy
train_accs = []
tests_accs = []
train_sizes = [0.2, 0.4, 0.6]
init_weights = model.get_weights()
for train_size in train_sizes:
  X_train_frac, _, y_train_frac, _ = train_test_split(X_train, y_train, train_size=train_size) # Split a fraction according to train_size
  model.set_weights(init_weights) # Make sure to use same random weights 
  model.fit(X_train_frac, y_train_frac, epochs=100, verbose=0, callbacks=[EarlyStopping(monitor='loss', patience=1)]) # Fit model on the training set fraction
  train_acc = model.evaluate(X_train_frac, y_train_frac, verbose=0)[1] # Get the accuracy for this training set fraction
  train_accs.append(train_acc)
  test_acc = model.evaluate(X_test, y_test, verbose=0)[1]
  test_accs.append(test_acc)
plt.plot(train_accs)
plt.plot(test_accs)

# k-fold Cross validation
from tensorflow.keras.wrappers.scikit_learn import KerasClassifier # Import sklearn wrapper from keras
model = KerasClassifier(build_fn=create_model, epochs=6, batch_size=16) # Create a model as a sklearn estimator
from sklearn.model_selection import cross_val_score
kfold = cross_val_score(model, X, y, cv=5) # Check how your keras model performs with 5 fold crossvalidation
kfold.mean() # Print the mean accuracy per fold
# Fine-tuning with RandomSearchCV
params = dict(optimizer=['sgd', 'adam'], epochs=3, batch_size=[5, 10, 20], activation=['relu','tanh'], nl=[1, 2, 9], nn=[128,256,1000])
random_search = RandomizedSearchCV(model, params_dist=params, cv=3)
random_search_results = random_search.fit(X, y)
random_search_results.best_score_
random_search_results.best_params_
```

# Chapter 3

### Visualizing overfitting

<center><img src="images/03.01.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.03.png"  style="width: 400px, height: 300px;"/></center>

### Unstable Learning curve

<center><img src="images/03.02.png"  style="width: 400px, height: 300px;"/></center>


### Which Activation Functions to use

- ReLU are a good first choice
- Sigmoids not recommended for deep models
- Tune with experimentation

<center><img src="images/03.04.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.06.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.07.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/03.08.png"  style="width: 400px, height: 300px;"/></center>

### Batch

- batch = training set
- mini-batch = subset of training set
- for each mini-batch, weight in each epoch is updated once
- helps faster training, less ram usage
- noise helps to reduce error and escape local minima
- Disadvantages: more iterations, needs adjusted batch size
<center><img src="images/03.09.png"  style="width: 400px, height: 300px;"/></center>


# Batch normalization

- helps avoid problems of activation functions and gradients
- makes sure inputs of the next layers are normalized
- Improves gradient .ow
- Allows higher learning rates
- Reduces dependence on weight initializations
- Acts as an unintended form of regularization
- Limits internal covariate shift
<center><img src="images/03.10.png"  style="width: 400px, height: 300px;"/></center>



# Tensors

- an array of numbers
- consists of n-dimensions
- eg : a rank-2 tensor means a 2-D array 

<center><img src="images/04.04.png"  style="width: 400px, height: 300px;"/></center>

### Keras tensor

```
# Import Keras backend
import tensorflow.keras.backend as K
# Get the input and output tensors of a model layer
inp = model.layers[0].input
out = model.layers[0].output
# Function that maps layer inputs to outputs
inp_to_out = K.function([inp], [out])
# We pass and input and get the output we'd get in that first layer
print(inp_to_out([X_train.values])) # make sure to convert into numpy arrays

# Visualize change in neuron outputs
import matplotlib.pyplot as plt
def plot():
  fig, ax = plt.subplots()
  plt.scatter(layer_output[:, 0], layer_output[:, 1],c = y_test,edgecolors='none')
  plt.title('Epoch: {}, Test Accuracy: {:3.1f} %'.format(i+1, test_accuracy * 100.0))
  plt.show()

for i in range(0, 21):
  	# Train model for 1 epoch
    h = model.fit(X_train, y_train, batch_size = 16, epochs = 1, verbose = 0)
    if i%4==0: 
      # Get the output of the first layer
      layer_output = inp_to_out([X_test.values])[0]
      
      # Evaluate model accuracy for this epoch
      test_accuracy = model.evaluate(X_test, y_test)[1] 
      
      # Plot 1st vs 2nd neuron output
      plot()

```

### Auto-encoders

- Dimensionality reduction:
    - Smaller dimensional space representation of our inputs.
- De-noising data:
    - If trained with clean data, irrelevant noise will be ,ltered out during reconstruction.
- Anomaly detection:
    - A poor reconstruction will result when the model is fed with unseen inputs.

<center><img src="images/04.01.png"  style="width: 400px, height: 300px;"/></center>


### Auto-encoder in keras

```
### Creating Auto-encoder
# Instantiate a sequential model
autoencoder = Sequential()
# Add a hidden layer of 4 neurons and an input layer of 100
autoencoder.add(Dense(4, input_shape=(100,), activation='relu')) # The hidden layer with 4 neurons will strictly compress the information to learn
# Add an output layer of 100 neurons
autoencoder.add(Dense(100, activation='sigmoid'))
# Compile your model with the appropiate loss
autoencoder.compile(optimizer='adam', loss='binary_crossentropy')

### A separate model to encode inputs
# Building a separate model to encode inputs
encoder = Sequential()
encoder.add(autoencoder.layers[0]) # Only add the first hidden layer with 4 neurons
# This will return what the four hidden layer neuron sees as features for the test data
encoder.predict(X_test)
```

### CNN

<center><img src="images/04.02.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.03.png"  style="width: 400px, height: 300px;"/></center>


### CNN in Keras

```
# Import Conv2D layer and Flatten from tensorflow keras layers
from tensorflow.keras.layers import Dense, Conv2D, Flatten
# Instantiate your model as usual
model = Sequential()
# Add a convolutional layer with 32 filters of size 3x3
model.add(Conv2D(filters=32, kernel_size=3, input_shape=(28, 28, 1), activation='relu'))
# Add another convolutional layer
model.add(Conv2D(filters=8, kernel_size=3, activation='relu'))
# Flatten the output of the previous layer
model.add(Flatten())
# End this multiclass model with 3 outputs and softmax
model.add(Dense(3, activation='softmax'))
```

### Using Resnet in keras

```
# Import image from keras preprocessing
from tensorflow.keras.preprocessing import image
# Import preprocess_input from tensorflow keras applications resnet50
from tensorflow.keras.applications.resnet50 import preprocess_input
# Load the image with the right target size for your model
img = image.load_img(img_path, target_size=(224, 224))
# Turn it into an array
img = image.img_to_array(img)
# Expand the dimensions so that it's understood by our network:
# img.shape turns from (224, 224, 3) into (1, 224, 224, 3)
img = np.expand_dims(img, axis=0)
# Pre-process the img in the same way training images were
img = preprocess_input(img)
# Import ResNet50 and decode_predictions from tensorflow.keras.applications.resnet50
from tensorflow.keras.applications.resnet50 import ResNet50, decode_predictions
# Instantiate a ResNet50 model with imagenet weights
model = ResNet50(weights='imagenet')
# Predict with ResNet50 on our img
preds = model.predict(img)
# Decode predictions and print it
print('Predicted:', decode_predictions(preds, top=1)[0])
```

# LSTM

- one type of RNN (RNN uses past predictions to infer new ones to solve problems where there is dependence on past inputs)
- used for problems related to sequencing. eg (translation, text generation, image captioning, musical composition, document summarization. etc)

<center><img src="images/04.05.png"  style="width: 400px, height: 300px;"/></center>
<center><img src="images/04.06.png"  style="width: 400px, height: 300px;"/></center>


### LSTM in keras (example of using on text data )

<center><img src="images/04.08.png"  style="width: 400px, height: 300px;"/></center>


```
text = 'Hi this is a small sentence'
# We choose a sequence length
seq_len = 3
# Split text into a list of words
words = text.split() # ['Hi', 'this', 'is', 'a', 'small', 'sentence']
# Make lines with 3 words like : ['Hi this is', 'this is a', 'is a small', 'a small sentence']
lines = []
for i in range(seq_len, len(words) + 1):
    line = ' '.join(words[i-seq_len:i])
    lines.append(line)
# Import Tokenizer from keras preprocessing text
from tensorflow.keras.preprocessing.text import Tokenizer
# Instantiate Tokenizer
tokenizer = Tokenizer()
# Fit it on the previous lines
tokenizer.fit_on_texts(lines)
# Turn the lines into numeric sequences
sequences = tokenizer.texts_to_sequences(lines) # array([[5, 3, 1], [3, 1, 2], [1, 2, 4], [2, 4, 6]])
print(tokenizer.index_word) # {1: 'is', 2: 'a', 3: 'this', 4: 'small', 5: 'hi', 6: 'sentence'}, we can use this to decode back original text

# Import Dense, LSTM and Embedding layers
from tensorflow.keras.layers import Dense, LSTM, Embedding
model = Sequential()
# Vocabulary size
vocab_size = len(tokenizer.index_word) + 1 # we are adding 1 since our encoding started from 1 and not 0, reserved for special characters
# Starting with an embedding layer (This is a layers that is specially required when we deal with categorical data like text in NLP to let the neural network understand the similarity between them)
# input_dim=size of unique tokens, input_length= length of input sequence, output_dim= dense vector embedding matrix columns
model.add(Embedding(input_dim=vocab_size, output_dim=8, input_length=2)) 
# Adding an LSTM layer
model.add(LSTM(8))
# Adding a Dense hidden layer
model.add(Dense(8, activation='relu'))
# Adding an output layer with softmax, last dense layer should have same number of inputs as input dimension of embedding layer
model.add(Dense(vocab_size, activation='softmax'))


#### Example 2
from tensorflow.keras.preprocessing.text import Tokenizer
# Split text into an array of words 
words = text.split()
# Make sentences of 4 words each, moving one word at a time
sentences = []
for i in range(4, len(words)):
  sentences.append(' '.join(words[i-4:i]))
# Instantiate a Tokenizer, then fit it on the sentences
tokenizer = Tokenizer()
tokenizer.fit_on_texts(sentences)
sequences = tokenizer.texts_to_sequences(sentences) # Turn sentences into a sequence of numbers
print("Sentences: \n {} \n Sequences: \n {}".format(sentences[:5],sequences[:5]))
vocab_size = len(tokenizer.index_word) + 1 
print(tokenizer.index_word)
from tensorflow.keras.layers import LSTM, Dense, Embedding # Import the Embedding, LSTM and Dense layer
model = Sequential()
# Add an Embedding layer with the right parameters
model.add(Embedding(input_dim = vocab_size, input_length = 3, output_dim = 8 )) # feed each neuron 3 words at a time
model.add(LSTM(32)) # Add a 32 unit LSTM layer
# Add a hidden Dense layer of 32 units and an output layer of vocab_size with softmax
model.add(Dense(32, activation='relu'))
model.add(Dense(vocab_size, activation='softmax'))
model.summary()
def predict_text(test_text, model = model):
  if len(test_text.split()) != 3:
    print('Text input should be 3 words!')
    return False
  # Turn the test_text into a sequence of numbers
  test_seq = tokenizer.texts_to_sequences([test_text])
  test_seq = np.array(test_seq)
  # Use the model passed as a parameter to predict the next word
  pred = model.predict(test_seq).argmax(axis = 1)[0]
  # Return the word that maps to the prediction
  return tokenizer.index_word[pred]
predict_text('meet revenge with')
```

### Text processing

```
# Remove non-letter characters
speech_df['text'] = speech_df['text'].str.replace('[^a-zA-Z]', ' ', regex=True)
# Standardize case
speech_df['text'] = speech_df['text'].str.lower()
# Generate Feature : Average length of word
speech_df['char_cnt'] = speech_df['text'].str.len()
speech_df['word_cnt'] = speech_df['text'].str.split().apply(len)
speech_df['avg_word_len'] = speech_df['char_cnt'] / speech_df['word_cnt']

# Generate Feature : tf-idf
from sklearn.feature_extraction.text import TfidfVectorizer
import nltk
# nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
from nltk.corpus import stopwords
vec = TfidfVectorizer(max_df=0.9, min_df=0.1, max_features=100, stop_words=stop_words) 

# Generate Feature : Bag of words / Word Count Vector
from sklearn.feature_extraction.text import CountVectorizer
vec = CountVectorizer(max_features=100, stop_words='english', min_df=0.1, max_df=0.9)

# Generate Feature : Introduce context with n-grams
vec = TfidfVectorizer(max_df=0.9, min_df=0.1, max_features=100, stop_words=stop_words, ngram_range = (2,2)) # Find context in 2 consecutive words
vec.fit(speech_df['text'])
transformed = vec.transform(speech_df['text'])
vec_df = pd.DataFrame(transformed.toarray(), columns=vec.get_feature_names_out()).add_prefix('Counts_')

# Sanity check : Find common words / patterns
vec_df.iloc[0].sort_values(ascending=False).head()
vec_df.sum().sort_values(ascending=False).head()

speech_df = pd.concat([speech_df, vec_df], axis=1, sort=False)

```